In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline  # Use imblearn's Pipeline
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import itertools

# Import oversampling techniques from imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [28, 234, 36, 215, 189, 37, 77, 26, 250, 188, 236, 181, 70, 55, 48, 67, 202, 244, 78, 65, 52, 237, 211, 57, 185, 229, 231, 219, 62, 220, 51, 200, 30, 242, 233, 198, 34, 25, 212, 205, 252, 32, 29, 248, 253, 75, 66, 247, 56, 58, 12, 221, 41, 197, 7, 63, 217, 1, 251, 199, 114, 49, 24, 50, 101, 80, 186, 232, 207, 130, 79, 115, 43, 243, 190, 137, 136, 108, 46, 214, 17, 10, 110, 88, 125, 182, 33, 203, 225, 68, 213, 227, 126, 93, 81, 14, 13, 104, 92, 42, 201, 129, 177, 122, 134, 133, 60, 19, 31, 9, 135, 98, 8, 45, 3, 47, 4, 6, 226, 116, 106, 90, 15, 105, 138, 18, 89, 84, 100, 44, 228, 131, 38, 112, 103, 96, 16, 127, 206, 117, 11, 102, 176, 128, 97, 0, 132, 5, 256, 61, 235, 87, 91, 193, 39, 111, 64, 180, 99, 95, 179, 35, 191, 246, 94, 238, 109, 22, 249, 187, 204, 245, 174, 53, 40, 86, 107, 119, 222, 239, 183, 157, 141, 123, 20, 196, 85, 83, 82, 167, 23, 139, 241, 72, 159, 2, 192, 175, 223, 54, 156, 73, 69, 208, 161, 120, 195, 158]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Create a pipeline with the sampler and classifier
pipeline = ImbPipeline([
    ('sampler', RandomOverSampler()),  # Placeholder, will be set by GridSearchCV
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the extended parameter grid as a list of dictionaries
param_grid = [
    {
        # Include options with and without a sampler
        'sampler': [SMOTE(), ADASYN(), RandomOverSampler()],
        # Only include sampling_strategy if sampler is not None
        'sampler__sampling_strategy': ['auto', 0.5, 0.75, 1.0],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    },
    {
        'sampler': [None],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    }
]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the number of parameter combinations
# This is the sum of the product of parameters in each dict in param_grid
n_param_combinations = 0
for grid in param_grid:
    n_combinations = 1
    for param in grid:
        n_combinations *= len(grid[param])
    n_param_combinations += n_combinations

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Calculate the total number of parameter combinations for the progress bar
total_fits = sum(
    np.prod([len(values) for values in grid.values()])
    for grid in param_grid
)

print("Starting Grid Search...")

# Wrap the fitting process with tqdm_joblib to show progress with the total number of fits
with tqdm_joblib(tqdm(desc="Grid Search Progress", total=total_fits)) as progress_bar:
    grid_search.fit(X_selected, Y)

print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

/home/azureuser/myenv/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Starting Grid Search...


  1%|▋                                                                            | 346/39936 [01:31<2:13:48,  4.93it/s]